In [1]:
import yaml 
from itertools import product 
from pathlib import Path
import os 
from swiss_roll import CONF_DIR
from swiss_roll import PROJECT_ROOT
from swiss_roll.config import Config, OptParams, CGDParams, \
    TrainParams, RunParams, SamplerParams, \
        HyperOptConfig, save_config, load_hyperoptconfig
from dataclasses import asdict

In [9]:
writeout_dir = CONF_DIR / "guidance_conf"
if not writeout_dir.exists():
    writeout_dir.mkdir(parents=True, exist_ok=True)

In [10]:
cfg = Config(
    opt=OptParams(),
    cgd=CGDParams(),
    train=TrainParams(),
    run=RunParams(),
    samp=SamplerParams(),
    )

writeout = PROJECT_ROOT / 'src' / 'swiss_roll' / 'example.yaml'
save_config(cfg, writeout)

In [13]:
reg_scale_bys = ['ctx_size', 'dataset_size']
reg_types = ['mse', 'cgd', 'None']
pareto_front = [True, False]
update_embeds = [True, False]
ctx_scales_idx = 0
reg_types_idx = 1
pareto_front_idx = 2
update_embeds_idx = 3 

search_space = list(product(reg_scale_bys, reg_types, pareto_front, update_embeds))

i = 0
for p in search_space: 
    cfg.cgd.reg_scale_by = p[ctx_scales_idx]
    cfg.run.reg_type = p[reg_types_idx]
    cfg.train.n_epochs = 10
    cfg.run.update_embeds = p[update_embeds_idx]

    if cfg.run.update_embeds and cfg.run.reg_type in ['mse', 'None']:
        continue 
    
    ho_cfg = HyperOptConfig(
        cfg=cfg,
        startup=10,
        seed=42,
        trials=60,
        timeout=12 * 60 * 60,
        pareto=p[pareto_front_idx],
    )
    
    cfg.run.run_id = f"{i:02d}"
    path = CONF_DIR / 'bayesopt_mse' / f'{cfg.run.run_id}.yaml'
    save_config(ho_cfg, path)
    i += 1  
print(i - 1) 

15


In [4]:
l2_lambda_values = [10**e for e in [-4, -3, -2, -1, 0, 1, 2, 3, 4]]
sigma_t_values   = [10**e for e in [-5, -3, -1, 0, 1, 3, 5]]
tau_t_values     = [10**e for e in [-5, -3, -1, 0, 1, 3, 5]]
reg_lambda       = [10**e for e in [0]]
ctx_size         = [512]       
use_ctx = [True]
batch_size = [128]
n_epochs = [100]
ctx_bounds = [1, 2.5, 5, 10, 25]

search_space = list(product(l2_lambda_values, 
                            sigma_t_values, 
                            tau_t_values, 
                            use_ctx,
                            ctx_size, 
                            reg_lambda,
                            batch_size,
                            n_epochs,
                            [ctx_bounds[1]]))

# Append just l2_lambda search for non-context
for l2 in l2_lambda_values:
    search_space.append((l2, 
                         None, 
                         None, 
                         False,
                         None, 
                         None,
                         128,
                         100,
                         ctx_bounds[1]))

# To check that configs registered properly
search_space.append((
    'l2_lambda', 
    'sigma_t_values',
    'tau_t_values',
    'use_ctx',
    'ctx_size', 
    'reg_lambda',
    128,
    100,
    'ctx_bounds',
))

print('Hyperparameter Combinations:', len(search_space))

Hyperparameter Combinations: 451


In [11]:
for root, dirs, paths in CONF_DIR.walk():
    for path in paths: 
        if path.endswith('.yaml'):
            os.remove(root / path) 

In [6]:
for i, conf in enumerate(search_space):
    padded_i = f"{i:04d}"
    config = {
        "l2_lambda": conf[0],
        "sigma_t": conf[1], 
        "tau_t": conf[2],
        "use_ctx": conf[3],
        "ctx_size": conf[4],
        "reg_lambda": conf[5],
        "batch_size": conf[6],
        "n_epochs": conf[7],
        "ctx_lower_bound": f'-{conf[8]}' if isinstance(conf[8], str) else -conf[8],
        "ctx_upper_bound": conf[8],
        "run_id": padded_i,
    }
    
    if i == len(search_space) - 1:
        for key, value in config.items():
            print(f"{key}: {value}")
        break  
    
    writeout = writeout_dir / f'{padded_i}.yaml'
    with open(writeout, 'w') as f:
        yaml.dump(config, f)


l2_lambda: l2_lambda
sigma_t: sigma_t_values
tau_t: tau_t_values
use_ctx: use_ctx
ctx_size: ctx_size
reg_lambda: reg_lambda
batch_size: 128
n_epochs: 100
ctx_lower_bound: -ctx_bounds
ctx_upper_bound: ctx_bounds
run_id: 0450


In [7]:
import subprocess
batching_script = './batching_script.sh'
cmd = f"bash {batching_script}"

batched_path = Path('./batched_configs')
for root, dirs, paths in batched_path.walk():
    for path in paths: 
        if path.startswith('batch'):
            os.remove(root / path) 


completed = subprocess.run(
    cmd, 
    shell=True, 
    check=True,
    capture_output=True,
    text=True
)